# 03 — Preprocessing

**Project:** Ecommerce-LLM-Finetuning
**Goal of this notebook:**
1. Load the raw dataset
2. Standardize column names to `instruction` / `response`
3. Clean text: remove HTML, URLs, noisy characters, normalize unicode/whitespace
4. Remove duplicate and empty rows
5. Filter by reasonable length bounds
6. Build final instruction-response pairs and save to `data/processed/`

This notebook is a thin wrapper around `src/preprocess.py` so all logic is
reusable, testable, and importable outside notebooks too.


In [1]:
import sys, os
sys.path.append(os.path.abspath(os.path.join('..')))
import pandas as pd

from src.config import Config
from src.preprocess import Preprocessor
from src.utils import get_logger, save_jsonl

logger = get_logger('notebook03')
cfg = Config()
cfg.ensure_dirs()


## 1. Load raw dataset

In [34]:
import sys, os
sys.path.append(os.path.abspath(".."))
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from src.config import Config
cfg = Config()

raw_path = cfg.raw_data_dir / "ecommerce_support_raw.csv"
df_raw = pd.read_csv(raw_path)
print(f"Loaded {len(df_raw):,} raw rows")
df_raw.head(3)


Loaded 26,872 raw rows


,flags,instruction,category,intent,response
0,B,question about cancelling order {{Order Number}},ORDER,cancel_order,I've understood you have a question regarding ...
1,BQZ,i have a question about cancelling oorder {{Or...,ORDER,cancel_order,I've been informed that you have a question ab...
2,BLQZ,i need help cancelling puchase {{Order Number}},ORDER,cancel_order,I can sense that you're seeking assistance wit...


## 2. Run the full cleaning pipeline

Steps (see `src/preprocess.py::Preprocessor.run`):
1. Standardize columns → `instruction`, `response`
2. Remove exact duplicates
3. Clean text (HTML, URLs, unicode, noisy chars, whitespace)
4. Remove empty rows post-clean
5. Filter rows outside min/max word bounds
6. Remove duplicates again (post-clean collisions)

In [36]:
from src.preprocess import Preprocessor
preprocessor = Preprocessor(min_words=2, max_words=400)
df_clean = preprocessor.run(df_raw, lowercase=False)

print(f"\nFinal clean dataset: {len(df_clean):,} rows (from {len(df_raw):,} raw rows)")
df_clean.head(5)


13:00:15 | INFO     | src.preprocess | Starting preprocessing on 26872 raw rows.
13:00:15 | INFO     | src.preprocess | Removed 0 duplicate rows (26872 -> 26872).
13:00:18 | INFO     | src.preprocess | Removed 0 empty rows (26872 -> 26872).
13:00:18 | INFO     | src.preprocess | Removed 5 rows outside length bounds (26872 -> 26867).
13:00:18 | INFO     | src.preprocess | Removed 0 duplicate rows (26867 -> 26867).
13:00:18 | INFO     | src.preprocess | Preprocessing complete: 26867 clean rows remain.

Final clean dataset: 26,867 rows (from 26,872 raw rows)


,instruction,response,category
0,question about cancelling order Order Number,I've understood you have a question regarding ...,ORDER
1,i have a question about cancelling oorder Orde...,I've been informed that you have a question ab...,ORDER
2,i need help cancelling puchase Order Number,I can sense that you're seeking assistance wit...,ORDER
3,I need to cancel purchase Order Number,I understood that you need assistance with can...,ORDER
4,"I cannot afford this order, cancel purchase Or...",I'm sensitive to the fact that you're facing f...,ORDER


## 3. Sanity checks

In [37]:
assert df_clean["instruction"].str.len().min() > 0, "Found empty instruction after cleaning!"
assert df_clean["response"].str.len().min() > 0, "Found empty response after cleaning!"
assert df_clean.duplicated(subset=["instruction", "response"]).sum() == 0, "Duplicates remain!"

print("All sanity checks passed ✅")
print("\nExample cleaned pair:")
print("INSTRUCTION:", df_clean.iloc[0]["instruction"])
print("RESPONSE:   ", df_clean.iloc[0]["response"])


All sanity checks passed ✅

Example cleaned pair:
INSTRUCTION: question about cancelling order Order Number
RESPONSE:    I've understood you have a question regarding canceling order Order Number , and I'm here to provide you with the information you need. Please go ahead and ask your question, and I'll do my best to assist you.


## 4. Save cleaned instruction-response pairs

In [39]:
from src.utils import save_jsonl
out_path = cfg.processed_data_dir / "cleaned_pairs.jsonl"
save_jsonl(df_clean.to_dict(orient="records"), out_path)

# Also keep a small human-readable CSV sample for quick spot-checks
sample_path = cfg.sample_data_dir / "cleaned_sample.csv"
df_clean.sample(min(50, len(df_clean)), random_state=cfg.seed).to_csv(sample_path, index=False)
print(f"Saved sample preview -> {sample_path}")


13:02:52 | INFO     | src.utils | Saved 26867 records -> C:\Users\Asus\Downloads\Ecommerce-LLM-Finetuning\Ecommerce-LLM-Finetuning\data\processed\cleaned_pairs.jsonl
Saved sample preview -> C:\Users\Asus\Downloads\Ecommerce-LLM-Finetuning\Ecommerce-LLM-Finetuning\data\sample\cleaned_sample.csv


## Summary

- Cleaned raw text (HTML/URL/unicode/noise/whitespace)
- Removed duplicate and empty rows
- Filtered by length bounds
- Saved clean instruction-response pairs to `data/processed/cleaned_pairs.jsonl`

Next: `04_prompt_formatting.ipynb` to convert these pairs into the training prompt format and tokenize them.